In [5]:
!pip install langchain-community llama-index pandas pypdf openai faiss-cpu sentence-transformers torch langchain-classic pydantic

In [15]:
import os 
import openai
import pandas as pd 
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import  RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS 
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableLambda
from IPython.display import display,Markdown
from langchain_core.prompts import ChatPromptTemplate

# native OpenAI client configured for OpenRouter
client = openai.OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="OPEN API KEY"  # Paste your full key here
)

#  custom function that converts LangChain inputs to an OpenRouter format
def openrouter_llm_call(prompt_input):
    if hasattr(prompt_input,"to_string"):
        prompt_text=prompt_input.to_string()
    else:    
        prompt_text = str(prompt_input)
        
    completion = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=[{"role": "user", "content": prompt_text}],
        temperature=0
    )
#    returns the lang chain message 
    return AIMessage(content=completion.choices[0].message.content)

#  Wrap it inside a LangChain Runnable so it acts as your regular model object
llm = RunnableLambda(openrouter_llm_call)

print("Custom OpenRouter LLM Gateway Initialized!")



Custom OpenRouter LLM Gateway Initialized!


In [7]:
# Load the file 
loader=PyPDFLoader("Attention is all you need.pdf")
raw_documents=loader.load()

#slice the text into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200) 
chunks= text_splitter.split_documents(raw_documents)

# load into pandas
chunks_data=[
    {"chunks_id": idx,"text":chunk.page_content[:150]+"..."}
    for idx,chunk in enumerate(chunks)
]
df_chunks=pd.DataFrame(chunks_data)
print(f"Successfully sliced document into {len(chunks)} chunks!")
df_chunks.head(5)

Successfully sliced document into 52 chunks!


,chunks_id,text
0,0,"Provided proper attribution is provided, Googl..."
1,1,mechanism. We propose a new simple network arc...
2,2,best models from the literature. We show that ...
3,3,efficient inference and visualizations. Lukasz...
4,4,"1 Introduction\nRecurrent neural networks, lon..."


In [21]:
print("Loading the local PyTorch-based model..")
embedding_model=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("Converting text chunks into vectors and building database...")
vector_db=FAISS.from_documents(chunks,embedding_model)

retriever=vector_db.as_retriever(search_kwargs={"k":7})

print("Database is loaded into memory and ready for search")

Loading the local PyTorch-based model..


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Converting text chunks into vectors and building database...
Database is loaded into memory and ready for search


In [ ]:


prompt = ChatPromptTemplate.from_messages([
("system", "You are an expert academic assistant.\nAnswer the user's question using ONLY the provided context below.If the answer is about document metadata (like authors), look at the first page elements rather than references at the end.\n\nContext:\n{context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human","{input}")
])
combine_docs_chain=create_stuff_documents_chain(llm,prompt)
rag_system=create_retrieval_chain(retriever,combine_docs_chain)

chat_history=[]

print("Interactive RAG system with memory ready (type 'exit' to stop )\n")

while True:
    user_question=input("You:")
    
    if user_question.lower()in['exit','quit']:
        print("Ending chat session. Goodbye!")
        break
    if not user_question.strip():
        continue

    response= rag_system.invoke({"input": user_question,
                                "chat_history":chat_history})
    print("-"* 50)
    display(Markdown(f"AI :{response['answer']}\n"))
    chat_history.append(HumanMessage(content=user_question))
    chat_history.append(AIMessage(content=response['answer']))
    


Interactive RAG system with memory ready (type 'exit' to stop )



You: who are the authors on the first page


--------------------------------------------------


AI :The authors on the first page are Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, and Illia Polosukhin.


You: what is the purpose of the positional encoding layer?


--------------------------------------------------


AI :The purpose of the positional encoding layer is to inject information about the relative or absolute position of the tokens in the sequence, allowing the model to make use of the order of the sequence since it contains no recurrence and no convolution.


You: What is the formula for Scaled Dot-Product Attention


--------------------------------------------------


AI :The formula for Scaled Dot-Product Attention is:

Attention(Q, K, V) = softmax(QK^T / √dk)V
